# WiDS Wildfire Survival Superblend


In [1]:
import os
import sys
import math
import time
import json
import glob
import re
import subprocess
import warnings
warnings.filterwarnings('ignore')


def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
        return
    except Exception:
        pass
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    except Exception:
        pass


for pkg, name in [
    ('lightgbm', 'lightgbm'),
    ('xgboost', 'xgboost'),
    ('catboost', 'catboost'),
    ('lifelines', 'lifelines'),
    ('scipy', 'scipy'),
]:
    _ensure(pkg, name)

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import HistGradientBoostingClassifier

try:
    import lightgbm as lgb
except Exception:
    lgb = None

try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None

try:
    from xgboost import XGBClassifier, DMatrix, train as xgb_train
except Exception:
    XGBClassifier = None
    DMatrix = None
    xgb_train = None

HAVE_LIFELINES = True
try:
    from lifelines import WeibullAFTFitter, LogNormalAFTFitter, LogLogisticAFTFitter
except Exception:
    HAVE_LIFELINES = False


SEED = 42
FAST_MODE = False
HORIZONS = [12, 24, 48, 72]
CV_SEEDS = [42, 52, 62, 72, 82] if FAST_MODE else [42, 52, 62, 72, 82, 92, 102, 112]
AFT_SEEDS = [131, 241, 353] if FAST_MODE else [131, 241, 353, 467, 577]
HAZARD_SEEDS = [701, 761, 821] if FAST_MODE else [701, 761, 821, 881, 941]
FOLDS = 5

NEAR_TRAIN_THR = {12: 7000.0, 24: 8500.0, 48: 12000.0, 72: 16000.0}
FAR_TRAIN_THR = {12: 3000.0, 24: 4500.0, 48: 6000.0, 72: 8000.0}
GATE_CFG = {12: (4500.0, 1200.0), 24: (5500.0, 1600.0), 48: (7000.0, 2300.0), 72: (9500.0, 3200.0)}

KNOWN_PRIOR_SCORES = {
    1: 0.96617,
    3: 0.96961,
    4: 0.97252,
    6: 0.97204,
    7: 0.97252,
    10: 0.97118,
    17: 0.97252,
}


def find_data_dir():
    explicit = [
        '/kaggle/input/widsworldwide-globaldathon26',
        '/kaggle/input/widsworldwide-globaldatathon26',
        '/kaggle/input/WiDSWorldWide_GlobalDathon26',
        '/kaggle/input/WiDSWorldWide_GlobalDatathon26',
        '.',
        './data',
        './input',
    ]
    needed = {'train.csv', 'test.csv', 'sample_submission.csv'}
    for path in explicit:
        if os.path.isdir(path) and needed.issubset(set(os.listdir(path))):
            return path
    for root in ['/kaggle/input', '.']:
        if not os.path.exists(root):
            continue
        for cur, _, files in os.walk(root):
            if needed.issubset(set(files)):
                return cur
    raise FileNotFoundError('Could not locate train.csv/test.csv/sample_submission.csv')


def compute_c_index(time_vals, event_vals, risk):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    risk = np.asarray(risk, dtype=float)
    conc = 0.0
    comp = 0.0
    for i in range(len(time_vals)):
        if event_vals[i] != 1:
            continue
        mask = time_vals[i] < time_vals
        comp += float(mask.sum())
        if mask.any():
            conc += float(np.sum(risk[i] > risk[mask]) + 0.5 * np.sum(risk[i] == risk[mask]))
    return conc / comp if comp else 0.5


def compute_brier(time_vals, event_vals, prob, horizon):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    prob = np.asarray(prob, dtype=float)
    valid = ~((event_vals == 0) & (time_vals < horizon))
    if valid.sum() == 0:
        return 0.25
    y_true = ((event_vals == 1) & (time_vals <= horizon)).astype(float)[valid]
    return float(np.mean((np.clip(prob[valid], 0.0, 1.0) - y_true) ** 2))


def hybrid_score(time_vals, event_vals, pred_matrix):
    pred_matrix = np.asarray(pred_matrix, dtype=float)
    p24, p48, p72 = pred_matrix[:, 1], pred_matrix[:, 2], pred_matrix[:, 3]
    risk = 0.3 * p24 + 0.4 * p48 + 0.3 * p72
    c_idx = compute_c_index(time_vals, event_vals, risk)
    b24 = compute_brier(time_vals, event_vals, p24, 24)
    b48 = compute_brier(time_vals, event_vals, p48, 48)
    b72 = compute_brier(time_vals, event_vals, p72, 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * c_idx + 0.7 * (1.0 - wb), c_idx, wb, (b24, b48, b72)


def make_binary_target(time_vals, event_vals, horizon):
    time_vals = np.asarray(time_vals, dtype=float)
    event_vals = np.asarray(event_vals, dtype=int)
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(int)
    return y, ~unknown


def compute_ipcw_weights(times, events, horizon):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    uniq = np.sort(np.unique(times))
    surv = np.ones(len(uniq), dtype=float)
    cur = 1.0
    for i, t in enumerate(uniq):
        at_risk = np.sum(times >= t)
        cens = np.sum((times == t) & (events == 0))
        if at_risk > 0:
            cur *= 1.0 - cens / at_risk
        surv[i] = cur

    def G(t):
        idx = np.searchsorted(uniq, t, side='right') - 1
        if idx >= 0:
            return max(float(surv[idx]), 0.01)
        return 1.0

    w = np.ones(len(times), dtype=float)
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            w[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            w[i] = 1.0 / G(horizon)
        else:
            w[i] = 0.0
    return w


def enforce_monotonicity(preds):
    out = np.clip(np.asarray(preds, dtype=float), 0.0, 1.0).copy()
    for j in range(1, out.shape[1]):
        out[:, j] = np.maximum(out[:, j], out[:, j - 1])
    return out


def soft_gate(dist_m, center, width):
    z = np.clip((center - np.asarray(dist_m, dtype=float)) / max(float(width), 1.0), -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-z))


def safe_predict_proba(model, X):
    p = model.predict_proba(X)
    if isinstance(p, list):
        p = np.asarray(p)
    return p[:, 1] if p.ndim == 2 else p


def fit_with_optional_weight(model, X, y, sample_weight=None):
    if sample_weight is None:
        model.fit(X, y)
        return model
    try:
        if isinstance(model, Pipeline):
            last_name = model.steps[-1][0]
            model.fit(X, y, **{f'{last_name}__sample_weight': sample_weight})
        else:
            model.fit(X, y, sample_weight=sample_weight)
    except Exception:
        model.fit(X, y)
    return model


def pct_rank(values, ref_values):
    ref_values = np.asarray(ref_values, dtype=float)
    if ref_values.size == 0:
        return np.zeros(len(values), dtype=float)
    ref_sorted = np.sort(ref_values)
    return np.searchsorted(ref_sorted, np.asarray(values, dtype=float), side='right') / ref_sorted.size


def create_features(df, fit_df=None):
    ref = fit_df if fit_df is not None else df
    r = df.copy()

    dist = r['dist_min_ci_0_5h'].clip(lower=1)
    speed = r['closing_speed_m_per_h']
    perims = r['num_perimeters_0_5h']
    area_first = r['area_first_ha']
    area_growth = r['area_growth_rate_ha_per_h']
    radial = r['radial_growth_rate_m_per_h'].clip(lower=0)
    align = r['alignment_abs']
    fire_radius = np.sqrt(np.maximum(area_first, 0.0) * 10000.0 / np.pi)
    closing_pos = speed.clip(lower=0)
    eff_close = closing_pos + radial

    r['dist_km'] = dist / 1000.0
    r['log_distance'] = np.log1p(dist)
    r['inv_distance'] = 1.0 / (r['dist_km'] + 0.1)
    r['inv_distance_sq'] = r['inv_distance'] ** 2
    r['sqrt_distance'] = np.sqrt(dist)
    r['dist_km_sq'] = r['dist_km'] ** 2
    r['dist_km_cb'] = r['dist_km'] ** 3

    r['fire_radius_km'] = fire_radius / 1000.0
    r['radius_to_dist'] = fire_radius / dist
    r['area_to_dist_ratio'] = area_first / (r['dist_km'] + 0.1)
    r['log_area_dist_ratio'] = np.log1p(area_first) - np.log1p(dist)

    r['has_movement'] = (perims > 1).astype(float)
    r['effective_closing_speed'] = eff_close
    r['eta_hours'] = np.where(closing_pos > 0.01, dist / closing_pos, 9999.0).clip(max=9999.0)
    r['eta_effective'] = np.where(eff_close > 0.01, dist / eff_close, 9999.0).clip(max=9999.0)
    r['log_eta'] = np.log1p(r['eta_hours'])
    r['log_eta_effective'] = np.log1p(r['eta_effective'])

    r['threat_score'] = align * speed / np.log1p(dist)
    r['threat_score_eff'] = align * eff_close / np.log1p(dist)
    r['threat_score_sq'] = r['threat_score_eff'] ** 2
    r['fire_urgency'] = perims * speed
    r['growth_intensity'] = area_growth * perims
    r['speed_alignment'] = speed * align
    r['effective_alignment'] = eff_close * align

    r['projected_advance_pos'] = np.maximum(r['projected_advance_m'], 0.0)
    r['closing_distance_pos'] = np.maximum(-r['dist_change_ci_0_5h'], 0.0)
    r['slope_toward'] = np.maximum(-r['dist_slope_ci_0_5h'], 0.0)
    r['accel_toward'] = np.maximum(-r['dist_accel_m_per_h2'], 0.0)

    r['zone_3km'] = (dist < 3000).astype(float)
    r['zone_near'] = (dist < 5000).astype(float)
    r['zone_warning'] = ((dist >= 5000) & (dist < 10000)).astype(float)
    r['zone_far'] = (dist >= 10000).astype(float)
    r['zone_20km'] = (dist < 20000).astype(float)

    r['is_summer'] = r['event_start_month'].isin([6, 7, 8]).astype(float)
    r['is_afternoon'] = ((r['event_start_hour'] >= 12) & (r['event_start_hour'] < 20)).astype(float)
    r['is_night'] = ((r['event_start_hour'] <= 6) | (r['event_start_hour'] >= 21)).astype(float)
    r['is_weekend'] = r['event_start_dayofweek'].isin([5, 6]).astype(float)

    r['hour_sin'] = np.sin(2 * np.pi * r['event_start_hour'] / 24.0)
    r['hour_cos'] = np.cos(2 * np.pi * r['event_start_hour'] / 24.0)
    r['month_sin'] = np.sin(2 * np.pi * (r['event_start_month'] - 1) / 12.0)
    r['month_cos'] = np.cos(2 * np.pi * (r['event_start_month'] - 1) / 12.0)
    r['dow_sin'] = np.sin(2 * np.pi * r['event_start_dayofweek'] / 7.0)
    r['dow_cos'] = np.cos(2 * np.pi * r['event_start_dayofweek'] / 7.0)

    ref_dist = ref['dist_min_ci_0_5h'].clip(lower=1).values
    ref_eff = ref['closing_speed_m_per_h'].clip(lower=0).values + ref['radial_growth_rate_m_per_h'].clip(lower=0).values
    ref_threat = ref['alignment_abs'].values * ref_eff / np.log1p(ref['dist_min_ci_0_5h'].clip(lower=1).values)

    r['dist_rank_global'] = pct_rank(dist.values, ref_dist)
    r['eff_rank_global'] = pct_rank(eff_close.values, ref_eff)
    r['threat_rank_global'] = pct_rank(r['threat_score_eff'].values, ref_threat)

    ref_near = ref['dist_min_ci_0_5h'].clip(lower=1).values < 5000
    cur_near = dist.values < 5000
    near_speed_rank = np.zeros(len(r), dtype=float)
    far_threat_rank = np.zeros(len(r), dtype=float)
    near_ref_eff = ref_eff[ref_near]
    far_ref_threat = ref_threat[~ref_near]
    if cur_near.any():
        near_speed_rank[cur_near] = pct_rank(eff_close.values[cur_near], near_ref_eff)
    if (~cur_near).any():
        far_threat_rank[~cur_near] = pct_rank(r['threat_score_eff'].values[~cur_near], far_ref_threat)
    r['near_speed_rank'] = near_speed_rank
    r['far_threat_rank'] = far_threat_rank

    r['wavefront_speed'] = r['effective_closing_speed'] + 0.35 * np.maximum(r['along_track_speed'], 0.0)
    r['eta_wavefront'] = np.where(r['wavefront_speed'] > 0.01, dist / r['wavefront_speed'], 9999.0).clip(max=9999.0)
    r['directed_growth'] = np.maximum(r['along_track_speed'], 0.0) + np.maximum(radial, 0.0) * align
    r['eta_directed'] = np.where(r['directed_growth'] > 0.01, dist / r['directed_growth'], 9999.0).clip(max=9999.0)
    r['near_miss_margin_m'] = dist - fire_radius - np.maximum(r['projected_advance_m'], 0.0)
    r['threat_gravity'] = (np.maximum(r['effective_closing_speed'], 0.0) + 1.0) * np.log1p(np.maximum(area_growth, 0.0) + 1.0) / np.power(r['dist_km'] + 0.25, 1.3)
    r['momentum_to_dist'] = (np.maximum(speed, 0.0) + np.maximum(radial, 0.0) + 1.0) / (r['dist_km'] + 0.5)
    r['close_then_grow'] = np.maximum(-r['dist_change_ci_0_5h'], 0.0) * np.log1p(np.maximum(area_growth, 0.0))
    r['radius_plus_displacement'] = fire_radius + np.maximum(r['centroid_displacement_m'], 0.0)
    r['reach_ratio'] = r['radius_plus_displacement'] / np.maximum(dist, 1.0)
    r['distance_resolution_ratio'] = r['dist_fit_r2_0_5h'] / (r['dt_first_last_0_5h'] + 0.1)
    r['movement_quality'] = (1.0 - r['low_temporal_resolution_0_5h']) * r['dist_fit_r2_0_5h'] * r['has_movement']
    r['close_speed_x_align'] = np.maximum(speed, 0.0) * (align ** 2)
    r['close_speed_x_area'] = np.maximum(speed, 0.0) * np.log1p(area_first)
    r['distance_decay_speed'] = np.exp(-r['dist_km'] / 10.0) * np.maximum(r['effective_closing_speed'], 0.0)
    r['distance_decay_threat'] = np.exp(-r['dist_km'] / 20.0) * r['threat_score_eff']
    r['track_balance'] = np.abs(r['along_track_speed']) / (np.abs(r['cross_track_component']) + 1.0)
    r['bearing_abs_cos'] = np.abs(r['spread_bearing_cos'])
    r['bearing_abs_sin'] = np.abs(r['spread_bearing_sin'])
    r['hour_x_month_sin'] = r['hour_sin'] * r['month_sin']
    r['hour_x_month_cos'] = r['hour_cos'] * r['month_cos']
    r['dist_x_month'] = r['dist_km'] * r['event_start_month']
    r['dist_x_hour'] = r['dist_km'] * r['event_start_hour']
    r['time_of_year_risk'] = 0.6 * r['month_sin'] + 0.4 * r['month_cos']
    r['fire_mass_proxy'] = np.sqrt(np.maximum(area_first, 0.0)) * (1.0 + np.maximum(area_growth, 0.0) / 100.0)
    r['mass_to_distance'] = r['fire_mass_proxy'] / (r['dist_km'] + 0.5)
    r['geometry_pressure'] = r['reach_ratio'] * (1.0 + align)
    r['closing_pressure'] = (1.0 / (r['eta_effective'] + 1.0)) * (1.0 + r['reach_ratio']) * (1.0 + 0.5 * align)
    r['timing_pressure'] = (1.0 / (r['eta_wavefront'] + 1.0)) + (1.0 / (r['eta_directed'] + 1.0))

    drop_cols = [
        'relative_growth_0_5h',
        'projected_advance_m',
        'centroid_displacement_m',
        'centroid_speed_m_per_h',
        'closing_speed_abs_m_per_h',
        'area_growth_abs_0_5h',
    ]
    r = r.drop(columns=[c for c in drop_cols if c in r.columns])
    r = r.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return r


def make_lgb_builder(params):
    def build(seed):
        if lgb is None:
            raise ImportError('lightgbm is unavailable')
        p = dict(params)
        p.update(dict(objective='binary', random_state=seed, n_jobs=-1, verbose=-1))
        return lgb.LGBMClassifier(**p)
    return build


def make_cat_builder(params):
    def build(seed):
        if CatBoostClassifier is None:
            raise ImportError('catboost is unavailable')
        p = dict(params)
        p.update(dict(loss_function='Logloss', eval_metric='Logloss', random_seed=seed, verbose=False, allow_writing_files=False))
        return CatBoostClassifier(**p)
    return build


def make_xgb_builder(params):
    def build(seed):
        if XGBClassifier is None:
            raise ImportError('xgboost classifier is unavailable')
        p = dict(params)
        p.update(dict(objective='binary:logistic', random_state=seed, eval_metric='logloss', n_jobs=-1))
        return XGBClassifier(**p)
    return build


def make_logit_builder(C=1.0):
    def build(seed):
        return Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('lr', LogisticRegression(C=C, solver='lbfgs', max_iter=4000)),
        ])
    return build


def make_hgb_builder(params):
    def build(seed):
        return Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('clf', HistGradientBoostingClassifier(random_state=seed, **params)),
        ])
    return build


def repeated_binary_model(X_train, X_test, times, events, horizon, build_model, seeds, n_splits=5, use_ipcw=True, calibrate='isotonic'):
    n_train = len(X_train)
    n_test = len(X_test)
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(n_test, dtype=float)

    for seed in seeds:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        seed_test = np.zeros(n_test, dtype=float)
        for tr_idx, va_idx in skf.split(X_train, events):
            y_tr, mask_tr = make_binary_target(times[tr_idx], events[tr_idx], horizon)
            keep_tr = np.where(mask_tr)[0]
            if len(keep_tr) == 0:
                pconst = float(((events == 1) & (times <= horizon)).mean())
                p_va = np.full(len(va_idx), pconst)
                p_te = np.full(n_test, pconst)
            else:
                xtr = X_train.iloc[tr_idx].iloc[keep_tr]
                ytr = y_tr[keep_tr]
                if np.unique(ytr).size < 2:
                    pconst = float(ytr.mean())
                    p_va = np.full(len(va_idx), pconst)
                    p_te = np.full(n_test, pconst)
                else:
                    model = build_model(seed)
                    sw = compute_ipcw_weights(times[tr_idx][keep_tr], events[tr_idx][keep_tr], horizon) if use_ipcw else None
                    fit_with_optional_weight(model, xtr, ytr, sw)
                    raw_tr = safe_predict_proba(model, xtr)
                    raw_va = safe_predict_proba(model, X_train.iloc[va_idx])
                    raw_te = safe_predict_proba(model, X_test)
                    if calibrate == 'isotonic' and np.unique(raw_tr).size >= 4:
                        cal = IsotonicRegression(out_of_bounds='clip')
                        try:
                            cal.fit(raw_tr, ytr, sample_weight=sw)
                        except Exception:
                            cal.fit(raw_tr, ytr)
                        p_va = cal.predict(raw_va)
                        p_te = cal.predict(raw_te)
                    elif calibrate == 'platt':
                        cal = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000)
                        try:
                            cal.fit(raw_tr.reshape(-1, 1), ytr, sample_weight=sw)
                        except Exception:
                            cal.fit(raw_tr.reshape(-1, 1), ytr)
                        p_va = cal.predict_proba(raw_va.reshape(-1, 1))[:, 1]
                        p_te = cal.predict_proba(raw_te.reshape(-1, 1))[:, 1]
                    else:
                        p_va = raw_va
                        p_te = raw_te
            oof[va_idx] += p_va
            cnt[va_idx] += 1.0
            seed_test += p_te
        test_pred += seed_test / n_splits

    oof = np.divide(oof, np.maximum(cnt, 1.0))
    test_pred = test_pred / len(seeds)
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)


def repeated_specialist_model(X_train, X_test, times, events, dist_train, dist_test, horizon, build_near, build_far, seeds, n_splits=5):
    n_train = len(X_train)
    n_test = len(X_test)
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(n_test, dtype=float)
    near_thr = NEAR_TRAIN_THR[horizon]
    far_thr = FAR_TRAIN_THR[horizon]
    gate_center, gate_width = GATE_CFG[horizon]

    for seed in seeds:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        seed_test = np.zeros(n_test, dtype=float)
        for tr_idx, va_idx in skf.split(X_train, events):
            y_tr, mask_tr = make_binary_target(times[tr_idx], events[tr_idx], horizon)
            xtr_all = X_train.iloc[tr_idx]
            xva = X_train.iloc[va_idx]
            tr_dist = dist_train[tr_idx]

            near_rows = np.where(mask_tr & (tr_dist < near_thr))[0]
            far_rows = np.where(mask_tr & (tr_dist >= far_thr))[0]
            glob_rows = np.where(mask_tr)[0]

            def fit_block(rows, build_model):
                if len(rows) < 16:
                    return None
                y_local = y_tr[rows]
                if np.unique(y_local).size < 2:
                    return None
                model = build_model(seed)
                sw_local = compute_ipcw_weights(times[tr_idx][rows], events[tr_idx][rows], horizon)
                fit_with_optional_weight(model, xtr_all.iloc[rows], y_local, sw_local)
                return model

            near_model = fit_block(near_rows, build_near)
            far_model = fit_block(far_rows, build_far)
            global_model = fit_block(glob_rows, build_far)

            def pred_from(model, Xp, default):
                if model is None:
                    return np.full(len(Xp), default, dtype=float)
                return safe_predict_proba(model, Xp)

            base_rate = float(((events[tr_idx] == 1) & (times[tr_idx] <= horizon)).mean())
            p_near_va = pred_from(near_model, xva, base_rate)
            p_far_va = pred_from(far_model, xva, base_rate)
            p_glob_va = pred_from(global_model, xva, base_rate)
            p_near_te = pred_from(near_model, X_test, base_rate)
            p_far_te = pred_from(far_model, X_test, base_rate)
            p_glob_te = pred_from(global_model, X_test, base_rate)

            gate_va = soft_gate(dist_train[va_idx], gate_center, gate_width)
            gate_te = soft_gate(dist_test, gate_center, gate_width)
            p_va = gate_va * p_near_va + (1.0 - gate_va) * p_far_va
            p_te = gate_te * p_near_te + (1.0 - gate_te) * p_far_te
            p_va = 0.80 * p_va + 0.20 * p_glob_va
            p_te = 0.80 * p_te + 0.20 * p_glob_te

            oof[va_idx] += p_va
            cnt[va_idx] += 1.0
            seed_test += p_te
        test_pred += seed_test / n_splits

    oof = np.divide(oof, np.maximum(cnt, 1.0))
    test_pred = test_pred / len(seeds)
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)


def aft_cdf(z, dist='normal'):
    z = np.asarray(z, dtype=float)
    if dist == 'normal':
        return 0.5 * (1.0 + np.vectorize(math.erf)(z / math.sqrt(2.0)))
    if dist == 'logistic':
        return 1.0 / (1.0 + np.exp(-z))
    if dist == 'extreme':
        return 1.0 - np.exp(-np.exp(z))
    raise ValueError(dist)


def xgb_aft_model(X_train, X_test, times, events, dist='normal', scale=1.0, seeds=None, n_splits=5, num_round=350):
    if xgb_train is None or DMatrix is None:
        raise ImportError('xgboost aft is unavailable')
    if seeds is None:
        seeds = AFT_SEEDS
    n_train = len(X_train)
    n_test = len(X_test)
    oof_loc = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    test_loc = np.zeros(n_test, dtype=float)
    Xtr_np = X_train.values.astype(np.float32)
    Xte_np = X_test.values.astype(np.float32)

    params = {
        'objective': 'survival:aft',
        'eval_metric': 'aft-nloglik',
        'aft_loss_distribution': dist,
        'aft_loss_distribution_scale': scale,
        'max_depth': 3,
        'eta': 0.035,
        'subsample': 0.85,
        'colsample_bynode': 0.85,
        'lambda': 1.0,
        'alpha': 0.08,
        'min_child_weight': 2.0,
        'tree_method': 'hist',
        'verbosity': 0,
    }

    for seed in seeds:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        seed_test = np.zeros(n_test, dtype=float)
        for tr_idx, va_idx in skf.split(Xtr_np, events):
            dtr = DMatrix(Xtr_np[tr_idx], enable_categorical=False)
            lower = np.asarray(times[tr_idx], dtype=float)
            upper = np.where(events[tr_idx] == 1, lower, np.inf)
            dtr.set_float_info('label_lower_bound', lower)
            dtr.set_float_info('label_upper_bound', upper)
            dva = DMatrix(Xtr_np[va_idx], enable_categorical=False)
            dte = DMatrix(Xte_np, enable_categorical=False)
            pars = dict(params)
            pars['seed'] = int(seed)
            booster = xgb_train(pars, dtr, num_boost_round=num_round)
            oof_loc[va_idx] += booster.predict(dva)
            cnt[va_idx] += 1.0
            seed_test += booster.predict(dte)
        test_loc += seed_test / n_splits

    oof_loc = np.divide(oof_loc, np.maximum(cnt, 1.0))
    test_loc = test_loc / len(seeds)
    oof_pred = np.column_stack([aft_cdf((np.log(float(h)) - oof_loc) / scale, dist=dist) for h in HORIZONS])
    test_pred = np.column_stack([aft_cdf((np.log(float(h)) - test_loc) / scale, dist=dist) for h in HORIZONS])
    return np.clip(oof_pred, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)


def maybe_lifelines_aft(train_proc, test_proc, times, events, seeds=None, n_splits=5):
    if not HAVE_LIFELINES:
        return None, None
    if seeds is None:
        seeds = [911, 1021, 1151] if FAST_MODE else [911, 1021, 1151, 1289]

    use_cols = [
        c for c in [
            'log_distance', 'dist_km', 'inv_distance', 'effective_closing_speed', 'eta_effective', 'eta_wavefront',
            'eta_directed', 'alignment_abs', 'reach_ratio', 'threat_score_eff', 'threat_gravity', 'momentum_to_dist',
            'closing_distance_pos', 'slope_toward', 'accel_toward', 'near_speed_rank', 'far_threat_rank',
            'distance_decay_threat', 'fire_radius_km', 'log1p_area_first', 'log1p_growth', 'month_sin', 'month_cos',
            'hour_sin', 'hour_cos', 'spread_bearing_sin', 'spread_bearing_cos', 'dist_fit_r2_0_5h',
            'low_temporal_resolution_0_5h', 'num_perimeters_0_5h', 'dt_first_last_0_5h'
        ] if c in train_proc.columns
    ]
    if len(use_cols) < 10:
        return None, None

    model_specs = [
        ('weibull', WeibullAFTFitter, 0.05),
        ('lognormal', LogNormalAFTFitter, 0.05),
        ('loglogistic', LogLogisticAFTFitter, 0.05),
    ]

    family_oof = []
    family_test = []

    for _, fitter_cls, penalizer in model_specs:
        n_train = len(train_proc)
        n_test = len(test_proc)
        oof = np.zeros((n_train, len(HORIZONS)), dtype=float)
        cnt = np.zeros(n_train, dtype=float)
        test_pred = np.zeros((n_test, len(HORIZONS)), dtype=float)
        success = False

        for seed in seeds:
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            seed_test = np.zeros((n_test, len(HORIZONS)), dtype=float)
            for tr_idx, va_idx in skf.split(train_proc, events):
                xtr = train_proc.iloc[tr_idx][use_cols].copy()
                xva = train_proc.iloc[va_idx][use_cols].copy()
                xte = test_proc[use_cols].copy()
                mu = xtr.mean(axis=0)
                sd = xtr.std(axis=0).replace(0, 1.0)
                xtr = (xtr - mu) / sd
                xva = (xva - mu) / sd
                xte = (xte - mu) / sd
                df_fit = xtr.copy()
                df_fit['T'] = times[tr_idx]
                df_fit['E'] = events[tr_idx]
                try:
                    fitter = fitter_cls(penalizer=penalizer)
                    fitter.fit(df_fit, duration_col='T', event_col='E', show_progress=False)
                    sf_va = fitter.predict_survival_function(xva, times=HORIZONS)
                    sf_te = fitter.predict_survival_function(xte, times=HORIZONS)
                    p_va = 1.0 - sf_va.T.values
                    p_te = 1.0 - sf_te.T.values
                    oof[va_idx] += p_va
                    cnt[va_idx] += 1.0
                    seed_test += p_te
                    success = True
                except Exception:
                    pass
            test_pred += seed_test / n_splits
        if success:
            oof = np.divide(oof, np.maximum(cnt[:, None], 1.0))
            test_pred = test_pred / len(seeds)
            family_oof.append(np.clip(oof, 0.0, 1.0))
            family_test.append(np.clip(test_pred, 0.0, 1.0))

    if not family_oof:
        return None, None
    weights = np.ones(len(family_oof), dtype=float) / len(family_oof)
    oof = np.tensordot(weights, np.stack(family_oof, axis=0), axes=(0, 0))
    te = np.tensordot(weights, np.stack(family_test, axis=0), axes=(0, 0))
    return np.clip(oof, 0.0, 1.0), np.clip(te, 0.0, 1.0)


def anchor_component(train_proc, test_proc, times, events, seeds=None, n_splits=5):
    if seeds is None:
        seeds = [1601, 1709, 1811] if FAST_MODE else [1601, 1709, 1811, 1913, 2017]
    n_train = len(train_proc)
    n_test = len(test_proc)

    def raw_score(df):
        s = (
            2.2 / (np.clip(df['eta_wavefront'].values, 0.0, 240.0) + 1.0)
            + 1.6 / (np.clip(df['eta_effective'].values, 0.0, 240.0) + 1.0)
            + 1.2 / (np.clip(df['eta_directed'].values, 0.0, 240.0) + 1.0)
            + 0.9 * np.clip(df['reach_ratio'].values, 0.0, 5.0)
            + 0.6 * np.clip(df['distance_decay_threat'].values, 0.0, None)
            + 0.4 * np.clip(df['near_speed_rank'].values, 0.0, 1.0)
            + 0.25 * np.clip(df['zone_near'].values, 0.0, 1.0)
            + 0.12 * np.clip(df['alignment_abs'].values, 0.0, 1.0)
        )
        return s.astype(float)

    score_train = raw_score(train_proc)
    score_test = raw_score(test_proc)
    oof = np.zeros((n_train, len(HORIZONS)), dtype=float)
    te = np.zeros((n_test, len(HORIZONS)), dtype=float)

    for h_idx, h in enumerate(HORIZONS):
        acc_oof = np.zeros(n_train, dtype=float)
        cnt = np.zeros(n_train, dtype=float)
        acc_te = np.zeros(n_test, dtype=float)
        for seed in seeds:
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            seed_test = np.zeros(n_test, dtype=float)
            for tr_idx, va_idx in skf.split(train_proc, events):
                y_tr, mask_tr = make_binary_target(times[tr_idx], events[tr_idx], h)
                tr_keep = np.where(mask_tr)[0]
                if len(tr_keep) < 12 or np.unique(y_tr[tr_keep]).size < 2:
                    pconst = float(((events[tr_idx] == 1) & (times[tr_idx] <= h)).mean())
                    p_va = np.full(len(va_idx), pconst)
                    p_te = np.full(n_test, pconst)
                else:
                    iso = IsotonicRegression(out_of_bounds='clip')
                    try:
                        iso.fit(score_train[tr_idx][tr_keep], y_tr[tr_keep], sample_weight=compute_ipcw_weights(times[tr_idx][tr_keep], events[tr_idx][tr_keep], h))
                    except Exception:
                        iso.fit(score_train[tr_idx][tr_keep], y_tr[tr_keep])
                    p_va = iso.predict(score_train[va_idx])
                    p_te = iso.predict(score_test)
                acc_oof[va_idx] += p_va
                cnt[va_idx] += 1.0
                seed_test += p_te
            acc_te += seed_test / n_splits
        oof[:, h_idx] = np.divide(acc_oof, np.maximum(cnt, 1.0))
        te[:, h_idx] = acc_te / len(seeds)
    return np.clip(oof, 0.0, 1.0), np.clip(te, 0.0, 1.0)


def expand_discrete_time(X, times=None, events=None, for_prediction=False):
    rows = []
    labels = []
    weights = []
    idx_pairs = []
    interval_specs = [(0.0, 12.0), (12.0, 24.0), (24.0, 48.0), (48.0, 72.0)]
    interval_weights = [1.0, 1.25, 1.45, 1.20]

    for i in range(len(X)):
        base = X.iloc[i]
        if for_prediction:
            for k, (start, end) in enumerate(interval_specs):
                rec = base.copy()
                rec['dt_interval_idx'] = k
                rec['dt_interval_start'] = start
                rec['dt_interval_end'] = end
                rec['dt_interval_width'] = end - start
                rec['dt_interval_mid'] = 0.5 * (start + end)
                rec['dt_log_interval_end'] = np.log1p(end)
                rows.append(rec)
                idx_pairs.append((i, k))
            continue

        t = float(times[i])
        e = int(events[i])
        for k, (start, end) in enumerate(interval_specs):
            if t <= start:
                break
            rec = base.copy()
            rec['dt_interval_idx'] = k
            rec['dt_interval_start'] = start
            rec['dt_interval_end'] = end
            rec['dt_interval_width'] = end - start
            rec['dt_interval_mid'] = 0.5 * (start + end)
            rec['dt_log_interval_end'] = np.log1p(end)
            if e == 1 and t <= end:
                rows.append(rec)
                labels.append(1)
                weights.append(interval_weights[k])
                idx_pairs.append((i, k))
                break
            if t >= end:
                rows.append(rec)
                labels.append(0)
                weights.append(interval_weights[k])
                idx_pairs.append((i, k))
                continue
            if e == 0 and t < end:
                break

    X_long = pd.DataFrame(rows)
    if for_prediction:
        return X_long, idx_pairs
    y_long = np.asarray(labels, dtype=int)
    w_long = np.asarray(weights, dtype=float)
    return X_long, y_long, w_long, idx_pairs


def discrete_hazard_component(X_train, X_test, times, events, seeds=None, n_splits=5):
    if seeds is None:
        seeds = HAZARD_SEEDS
    if lgb is None:
        raise ImportError('lightgbm is required for discrete hazard model')
    n_train = len(X_train)
    n_test = len(X_test)
    oof = np.zeros((n_train, len(HORIZONS)), dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    te = np.zeros((n_test, len(HORIZONS)), dtype=float)

    pred_long_test, test_pairs = expand_discrete_time(X_test, for_prediction=True)

    params = dict(
        n_estimators=220 if FAST_MODE else 320,
        learning_rate=0.035,
        num_leaves=19,
        max_depth=4,
        min_child_samples=12,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.25,
        reg_lambda=1.0,
        objective='binary',
        n_jobs=-1,
        verbose=-1,
    )

    def long_to_cumprob(haz):
        haz = np.clip(haz, 1e-6, 1.0 - 1e-6)
        surv = np.cumprod(1.0 - haz, axis=1)
        return 1.0 - surv

    for seed in seeds:
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        seed_test = np.zeros((n_test, len(HORIZONS)), dtype=float)
        for tr_idx, va_idx in skf.split(X_train, events):
            xtr_long, ytr_long, wtr_long, _ = expand_discrete_time(X_train.iloc[tr_idx], times[tr_idx], events[tr_idx], for_prediction=False)
            xva_long, va_pairs = expand_discrete_time(X_train.iloc[va_idx], for_prediction=True)
            model = lgb.LGBMClassifier(random_state=seed, **params)
            fit_with_optional_weight(model, xtr_long, ytr_long, wtr_long)
            haz_va = safe_predict_proba(model, xva_long)
            haz_te = safe_predict_proba(model, pred_long_test)
            haz_va_mat = np.zeros((len(va_idx), len(HORIZONS)), dtype=float)
            haz_te_mat = np.zeros((n_test, len(HORIZONS)), dtype=float)
            for value, (row_i, int_k) in zip(haz_va, va_pairs):
                haz_va_mat[row_i, int_k] = value
            for value, (row_i, int_k) in zip(haz_te, test_pairs):
                haz_te_mat[row_i, int_k] = value
            p_va = long_to_cumprob(haz_va_mat)
            p_te = long_to_cumprob(haz_te_mat)
            oof[va_idx] += p_va
            cnt[va_idx] += 1.0
            seed_test += p_te
        te += seed_test / n_splits
    oof = np.divide(oof, np.maximum(cnt[:, None], 1.0))
    te = te / len(seeds)
    return np.clip(oof, 0.0, 1.0), np.clip(te, 0.0, 1.0)


def search_simplex_weights(pred_list, times, events, horizon, prior=None, n_iter=5000, seed=42, reg=0.0025):
    m = len(pred_list)
    if m == 1:
        return np.array([1.0], dtype=float)
    valid = ~((events == 0) & (times < horizon))
    y = ((events == 1) & (times <= horizon)).astype(float)[valid]
    P = np.column_stack([np.asarray(p, dtype=float)[valid] for p in pred_list])
    if prior is None:
        prior = np.ones(m, dtype=float) / m
    prior = np.asarray(prior, dtype=float)
    prior = np.clip(prior, 1e-6, None)
    prior = prior / prior.sum()

    best_w = prior.copy()
    best_loss = float(np.mean((P.dot(best_w) - y) ** 2) + reg * np.sum((best_w - prior) ** 2))
    rng = np.random.default_rng(seed)
    candidates = [prior, np.ones(m) / m]
    candidates += [np.eye(m)[i] for i in range(m)]
    for i in range(m):
        for j in range(i + 1, m):
            for a in [0.2, 0.35, 0.5, 0.65, 0.8]:
                w = np.zeros(m, dtype=float)
                w[i] = a
                w[j] = 1.0 - a
                candidates.append(w)
    alpha = 1.0 + 28.0 * prior
    for _ in range(n_iter):
        candidates.append(rng.dirichlet(alpha))
    for w in candidates:
        p = np.clip(P.dot(w), 0.0, 1.0)
        loss = float(np.mean((p - y) ** 2) + reg * np.sum((w - prior) ** 2))
        if loss < best_loss:
            best_loss = loss
            best_w = w.copy()
    return best_w / best_w.sum()


def family_blend(pred_store, times, events, priors_by_h=None, n_iter=3500, seed=42):
    names = list(pred_store.keys())
    oof_stack = [pred_store[k][0] for k in names]
    test_stack = [pred_store[k][1] for k in names]
    out_oof = np.zeros_like(oof_stack[0])
    out_test = np.zeros_like(test_stack[0])
    weight_matrix = {}

    for h_idx, h in enumerate(HORIZONS):
        pred_list_oof = [arr[:, h_idx] for arr in oof_stack]
        prior = None
        if priors_by_h is not None and h in priors_by_h:
            prior = np.asarray([priors_by_h[h].get(name, 0.0) for name in names], dtype=float)
            if prior.sum() <= 0:
                prior = None
        w = search_simplex_weights(pred_list_oof, times, events, h, prior=prior, n_iter=n_iter, seed=seed + h_idx)
        weight_matrix[h] = dict(zip(names, w.tolist()))
        out_oof[:, h_idx] = np.sum([w[i] * oof_stack[i][:, h_idx] for i in range(len(names))], axis=0)
        out_test[:, h_idx] = np.sum([w[i] * test_stack[i][:, h_idx] for i in range(len(names))], axis=0)

    return np.clip(out_oof, 0.0, 1.0), np.clip(out_test, 0.0, 1.0), weight_matrix


def calibrate_matrix(oof_pred, test_pred, times, events):
    oof_pred = np.clip(np.asarray(oof_pred, dtype=float), 0.0, 1.0)
    test_pred = np.clip(np.asarray(test_pred, dtype=float), 0.0, 1.0)
    out_oof = oof_pred.copy()
    out_test = test_pred.copy()

    for h_idx, h in enumerate(HORIZONS):
        valid = ~((events == 0) & (times < h))
        y = ((events == 1) & (times <= h)).astype(float)[valid]
        x = out_oof[valid, h_idx]
        best_oof = out_oof[:, h_idx].copy()
        best_test = out_test[:, h_idx].copy()
        best_loss = compute_brier(times, events, best_oof, h)

        if np.unique(x).size >= 6:
            try:
                iso = IsotonicRegression(out_of_bounds='clip')
                iso.fit(x, y)
                cand_oof = iso.predict(out_oof[:, h_idx])
                cand_test = iso.predict(out_test[:, h_idx])
                cand_loss = compute_brier(times, events, cand_oof, h)
                if cand_loss < best_loss:
                    best_oof, best_test, best_loss = cand_oof, cand_test, cand_loss
            except Exception:
                pass

        for alpha in [0.75, 0.85, 0.95, 1.05, 1.15, 1.30, 1.50]:
            cand_oof = np.power(np.clip(out_oof[:, h_idx], 1e-8, 1.0), alpha)
            cand_test = np.power(np.clip(out_test[:, h_idx], 1e-8, 1.0), alpha)
            cand_loss = compute_brier(times, events, cand_oof, h)
            if cand_loss < best_loss:
                best_oof, best_test, best_loss = cand_oof, cand_test, cand_loss
            cand_oof = 1.0 - np.power(np.clip(1.0 - out_oof[:, h_idx], 1e-8, 1.0), alpha)
            cand_test = 1.0 - np.power(np.clip(1.0 - out_test[:, h_idx], 1e-8, 1.0), alpha)
            cand_loss = compute_brier(times, events, cand_oof, h)
            if cand_loss < best_loss:
                best_oof, best_test, best_loss = cand_oof, cand_test, cand_loss

        out_oof[:, h_idx] = np.clip(best_oof, 0.0, 1.0)
        out_test[:, h_idx] = np.clip(best_test, 0.0, 1.0)

    return np.clip(out_oof, 0.0, 1.0), np.clip(out_test, 0.0, 1.0)


def search_final_weight_matrix(component_store, times, events, n_iter=14000, seed=42):
    names = list(component_store.keys())
    oof_stack = np.stack([component_store[n][0] for n in names], axis=0)
    test_stack = np.stack([component_store[n][1] for n in names], axis=0)
    m = len(names)

    prior_map = {
        'global_cls': np.array([0.30, 0.28, 0.24, 0.22], dtype=float),
        'zone': np.array([0.24, 0.23, 0.18, 0.15], dtype=float),
        'aft': np.array([0.12, 0.18, 0.28, 0.34], dtype=float),
        'hazard': np.array([0.20, 0.19, 0.16, 0.12], dtype=float),
        'anchor': np.array([0.14, 0.10, 0.08, 0.07], dtype=float),
        'lifelines': np.array([0.00, 0.02, 0.06, 0.10], dtype=float),
    }
    prior = np.column_stack([prior_map.get(name, np.full(4, 1.0 / max(m, 1), dtype=float)) for name in names]).T
    prior = np.clip(prior, 1e-6, None)
    prior = prior / prior.sum(axis=0, keepdims=True)

    def apply_weight_matrix(W, stack):
        out = np.zeros(stack.shape[1:], dtype=float)
        for h_idx in range(4):
            out[:, h_idx] = np.tensordot(W[:, h_idx], stack[:, :, h_idx], axes=(0, 0))
        return enforce_monotonicity(np.clip(out, 0.0, 1.0))

    best_W = prior.copy()
    best_oof = apply_weight_matrix(best_W, oof_stack)
    best_score = hybrid_score(times, events, best_oof)[0] - 0.03 * compute_brier(times, events, best_oof[:, 0], 12)
    rng = np.random.default_rng(seed)
    alphas = 1.0 + 30.0 * prior

    candidates = [prior.copy(), np.full_like(prior, 1.0 / m)]
    for k in range(m):
        W = np.zeros_like(prior)
        W[k, :] = 1.0
        candidates.append(W)

    for _ in range(n_iter):
        W = np.zeros_like(prior)
        for h_idx in range(4):
            W[:, h_idx] = rng.dirichlet(alphas[:, h_idx])
        candidates.append(W)

    for W in candidates:
        pred = apply_weight_matrix(W, oof_stack)
        score = hybrid_score(times, events, pred)[0] - 0.03 * compute_brier(times, events, pred[:, 0], 12) - 0.002 * np.sum((W - prior) ** 2)
        if score > best_score:
            best_score = score
            best_W = W.copy()
            best_oof = pred.copy()

    best_test = apply_weight_matrix(best_W, test_stack)
    return best_oof, best_test, best_W, names


def optional_prior_consensus(sample_sub):
    paths = []
    for root in ['/kaggle/input', '.']:
        if os.path.exists(root):
            paths.extend(glob.glob(os.path.join(root, '**', 'samplecsv_sub*.csv'), recursive=True))
    paths = sorted(set(paths))
    if not paths:
        return None

    preds = []
    weights = []
    seen = []
    target_ids = sample_sub['event_id'].astype(str).tolist()

    for path in paths:
        m = re.search(r'samplecsv_sub(\d+)\.csv$', os.path.basename(path))
        if not m:
            continue
        sub_no = int(m.group(1))
        if sub_no not in KNOWN_PRIOR_SCORES:
            continue
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        cols = ['event_id', 'prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']
        if not set(cols).issubset(df.columns):
            continue
        df = df[cols].copy()
        if df['event_id'].astype(str).tolist() != target_ids:
            df = sample_sub[['event_id']].merge(df, on='event_id', how='left')
            if df[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']].isna().any().any():
                continue
        arr = df[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']].values.astype(float)
        duplicate = False
        for prev in seen:
            if np.max(np.abs(arr - prev)) < 1e-10:
                duplicate = True
                break
        if duplicate:
            continue
        seen.append(arr)
        preds.append(arr)
        weights.append(max(KNOWN_PRIOR_SCORES[sub_no] - 0.955, 1e-4) ** 2)

    if not preds:
        return None
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    blend = np.tensordot(weights, np.stack(preds, axis=0), axes=(0, 0))
    return np.clip(blend, 0.0, 1.0)


def main():
    data_dir = find_data_dir()
    train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
    test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))
    sample_sub = pd.read_csv(os.path.join(data_dir, 'sample_submission.csv'))

    train_proc = create_features(train_df, fit_df=train_df)
    test_proc = create_features(test_df, fit_df=train_df)

    feature_cols = [c for c in train_proc.columns if c not in ['event_id', 'time_to_hit_hours', 'event']]
    X_train = train_proc[feature_cols].copy()
    X_test = test_proc[feature_cols].copy()
    times = train_df['time_to_hit_hours'].values.astype(float)
    events = train_df['event'].astype(int).values
    dist_train = train_df['dist_min_ci_0_5h'].values.astype(float)
    dist_test = test_df['dist_min_ci_0_5h'].values.astype(float)

    print('DATA_DIR =', data_dir)
    print('train shape =', train_df.shape, '| test shape =', test_df.shape)
    print('events =', train_df['event'].value_counts().to_dict())

    global_builders = {}
    if lgb is not None:
        global_builders['lgb'] = make_lgb_builder(dict(
            n_estimators=260 if FAST_MODE else 360,
            learning_rate=0.032,
            num_leaves=17,
            max_depth=4,
            min_child_samples=10,
            subsample=0.82,
            colsample_bytree=0.82,
            reg_alpha=0.22,
            reg_lambda=0.95,
        ))
    if CatBoostClassifier is not None:
        global_builders['cat'] = make_cat_builder(dict(
            depth=4,
            learning_rate=0.034,
            iterations=260 if FAST_MODE else 360,
            l2_leaf_reg=5.0,
            bootstrap_type='Bernoulli',
            subsample=0.82,
        ))
    if XGBClassifier is not None:
        global_builders['xgb'] = make_xgb_builder(dict(
            n_estimators=260 if FAST_MODE else 340,
            max_depth=3,
            learning_rate=0.035,
            subsample=0.84,
            colsample_bytree=0.82,
            reg_lambda=1.0,
            reg_alpha=0.08,
            min_child_weight=2.0,
            gamma=0.0,
        ))
    global_builders['logit'] = make_logit_builder(C=0.75)
    global_builders['hgb'] = make_hgb_builder(dict(
        max_depth=3,
        learning_rate=0.040,
        max_iter=180 if FAST_MODE else 240,
        min_samples_leaf=8,
        l2_regularization=0.15,
    ))

    global_store = {}
    for name, builder in global_builders.items():
        print(f'Building global classifier family: {name}')
        oof = np.zeros((len(train_df), 4), dtype=float)
        te = np.zeros((len(test_df), 4), dtype=float)
        for h_idx, h in enumerate(HORIZONS):
            oo, tt = repeated_binary_model(X_train, X_test, times, events, h, builder, seeds=CV_SEEDS, n_splits=FOLDS, use_ipcw=True, calibrate='isotonic')
            oof[:, h_idx] = oo
            te[:, h_idx] = tt
        oof = enforce_monotonicity(oof)
        te = enforce_monotonicity(te)
        global_store[name] = (oof, te)
        print('  hybrid =', round(hybrid_score(times, events, oof)[0], 6))

    global_priors = {
        12: {'lgb': 0.28, 'cat': 0.24, 'xgb': 0.18, 'logit': 0.16, 'hgb': 0.14},
        24: {'lgb': 0.28, 'cat': 0.24, 'xgb': 0.18, 'logit': 0.16, 'hgb': 0.14},
        48: {'lgb': 0.26, 'cat': 0.22, 'xgb': 0.18, 'logit': 0.18, 'hgb': 0.16},
        72: {'lgb': 0.24, 'cat': 0.20, 'xgb': 0.18, 'logit': 0.20, 'hgb': 0.18},
    }
    global_cls_oof, global_cls_test, _ = family_blend(global_store, times, events, priors_by_h=global_priors, n_iter=2800 if FAST_MODE else 4200, seed=77)
    print('global_cls hybrid =', round(hybrid_score(times, events, enforce_monotonicity(global_cls_oof))[0], 6))

    if CatBoostClassifier is not None:
        near_builder = make_cat_builder(dict(
            depth=4,
            learning_rate=0.036,
            iterations=240 if FAST_MODE else 320,
            l2_leaf_reg=6.0,
            bootstrap_type='Bernoulli',
            subsample=0.85,
        ))
    else:
        near_builder = make_logit_builder(C=1.0)
    if lgb is not None:
        far_builder = make_lgb_builder(dict(
            n_estimators=220 if FAST_MODE else 300,
            learning_rate=0.034,
            num_leaves=15,
            max_depth=4,
            min_child_samples=12,
            subsample=0.84,
            colsample_bytree=0.84,
            reg_alpha=0.18,
            reg_lambda=0.9,
        ))
    else:
        far_builder = make_logit_builder(C=1.0)

    zone_oof = np.zeros((len(train_df), 4), dtype=float)
    zone_test = np.zeros((len(test_df), 4), dtype=float)
    for h_idx, h in enumerate(HORIZONS):
        print(f'Building zone specialist: horizon={h}')
        oo, tt = repeated_specialist_model(X_train, X_test, times, events, dist_train, dist_test, h, near_builder, far_builder, seeds=CV_SEEDS, n_splits=FOLDS)
        zone_oof[:, h_idx] = oo
        zone_test[:, h_idx] = tt
    zone_oof = enforce_monotonicity(zone_oof)
    zone_test = enforce_monotonicity(zone_test)
    print('zone hybrid =', round(hybrid_score(times, events, zone_oof)[0], 6))

    anchor_oof, anchor_test = anchor_component(train_proc, test_proc, times, events)
    anchor_oof = enforce_monotonicity(anchor_oof)
    anchor_test = enforce_monotonicity(anchor_test)
    print('anchor hybrid =', round(hybrid_score(times, events, anchor_oof)[0], 6))

    hazard_oof, hazard_test = discrete_hazard_component(X_train, X_test, times, events)
    hazard_oof = enforce_monotonicity(hazard_oof)
    hazard_test = enforce_monotonicity(hazard_test)
    print('hazard hybrid =', round(hybrid_score(times, events, hazard_oof)[0], 6))

    aft_store = {}
    if xgb_train is not None and DMatrix is not None:
        aft_cfgs = [
            ('aft_normal_08', 'normal', 0.8),
            ('aft_normal_10', 'normal', 1.0),
            ('aft_logistic_10', 'logistic', 1.0),
            ('aft_extreme_10', 'extreme', 1.0),
        ]
        for name, dist_name, scale in aft_cfgs:
            try:
                print(f'Building AFT model: {name}')
                oo, tt = xgb_aft_model(X_train, X_test, times, events, dist=dist_name, scale=scale)
                oo, tt = calibrate_matrix(oo, tt, times, events)
                oo = enforce_monotonicity(oo)
                tt = enforce_monotonicity(tt)
                aft_store[name] = (oo, tt)
                print('  hybrid =', round(hybrid_score(times, events, oo)[0], 6))
            except Exception as ex:
                print('  skip', name, '->', repr(ex))

    aft_oof = None
    aft_test = None
    if aft_store:
        aft_priors = {
            12: {k: 1.0 / len(aft_store) for k in aft_store},
            24: {k: 1.0 / len(aft_store) for k in aft_store},
            48: {k: 1.0 / len(aft_store) for k in aft_store},
            72: {k: 1.0 / len(aft_store) for k in aft_store},
        }
        aft_oof, aft_test, _ = family_blend(aft_store, times, events, priors_by_h=aft_priors, n_iter=2500 if FAST_MODE else 3800, seed=191)
        aft_oof = enforce_monotonicity(aft_oof)
        aft_test = enforce_monotonicity(aft_test)
        print('aft family hybrid =', round(hybrid_score(times, events, aft_oof)[0], 6))

    life_oof, life_test = maybe_lifelines_aft(train_proc, test_proc, times, events)
    if life_oof is not None:
        life_oof, life_test = calibrate_matrix(life_oof, life_test, times, events)
        life_oof = enforce_monotonicity(life_oof)
        life_test = enforce_monotonicity(life_test)
        print('lifelines hybrid =', round(hybrid_score(times, events, life_oof)[0], 6))

    component_store = {
        'global_cls': (global_cls_oof, global_cls_test),
        'zone': (zone_oof, zone_test),
        'hazard': (hazard_oof, hazard_test),
        'anchor': (anchor_oof, anchor_test),
    }
    if aft_oof is not None:
        component_store['aft'] = (aft_oof, aft_test)
    if life_oof is not None:
        component_store['lifelines'] = (life_oof, life_test)

    blend_oof, blend_test, final_W, final_names = search_final_weight_matrix(component_store, times, events, n_iter=7000 if FAST_MODE else 12000, seed=313)
    blend_oof, blend_test = calibrate_matrix(blend_oof, blend_test, times, events)
    blend_oof = enforce_monotonicity(blend_oof)
    blend_test = enforce_monotonicity(blend_test)

    prior_consensus = optional_prior_consensus(sample_sub)
    if prior_consensus is not None:
        print('Optional prior submission consensus found -> applying small hedge blend')
        blend_test = enforce_monotonicity(0.96 * blend_test + 0.04 * prior_consensus)

    final_score, final_c, final_wb, final_b = hybrid_score(times, events, blend_oof)
    print('FINAL OOF HYBRID =', round(final_score, 6))
    print('FINAL OOF C-INDEX =', round(final_c, 6))
    print('FINAL OOF WB =', round(final_wb, 6), '| BRIERS =', tuple(round(x, 6) for x in final_b))
    print('FINAL COMPONENTS =', final_names)
    print('FINAL WEIGHT MATRIX =')
    for h_idx, h in enumerate(HORIZONS):
        print(h, {final_names[i]: round(float(final_W[i, h_idx]), 4) for i in range(len(final_names))})

    submission = sample_sub[['event_id']].copy()
    submission[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']] = enforce_monotonicity(blend_test)
    submission = submission[['event_id', 'prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']]
    assert submission['event_id'].tolist() == sample_sub['event_id'].tolist()
    assert submission['event_id'].is_unique
    assert np.isfinite(submission[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']].values).all()
    assert ((submission[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']] >= 0.0) & (submission[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']] <= 1.0)).all().all()
    assert (submission['prob_12h'] <= submission['prob_24h']).all()
    assert (submission['prob_24h'] <= submission['prob_48h']).all()
    assert (submission['prob_48h'] <= submission['prob_72h']).all()

    submission.to_csv('submission.csv', index=False)
    pd.DataFrame({'metric': ['hybrid', 'c_index', 'weighted_brier', 'brier_24', 'brier_48', 'brier_72'], 'value': [final_score, final_c, final_wb, final_b[0], final_b[1], final_b[2]]}).to_csv('cv_metrics.csv', index=False)
    print('\nSaved submission.csv')
    print(submission.head())


if __name__ == '__main__':
    main()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 6.2 MB/s eta 0:00:00
DATA_DIR = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
train shape = (221, 37) | test shape = (95, 35)
events = {0: 152, 1: 69}
Building global classifier family: lgb
  hybrid = 0.964959
Building global classifier family: cat
  hybrid = 0.962378
Building global classifier family: xgb
  hybrid = 0.95718
Building global classifier family: logit
  hybrid = 0.967609
Building global classifier family: hgb
  hybrid = 0.965899
global_cls hybrid = 0.96808
Building zone specialist: horizon=12
Building zone specialist: horizon=24
Building zone specialist: horizon=48
Building zone specialist: horizon=72
zone hybrid = 0.955643
anchor hybrid = 0.959492
hazard hybrid = 0.970976
Building AFT model: aft_normal_08
  hybrid = 0.926078
Building AFT model: aft_normal_10
  hybrid = 0.929004
Building AFT model: aft_logistic_10
  hybrid =